In [ ]:
import datasets
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import random
import torch.nn.functional as F

In [ ]:
import re


def format_prompt(question, options):
    options = "\n".join([f"{i+1}. {opt}" for i, opt in enumerate(options)])
    prompt = f"""Answer the following question accurately with one of the options provided. Answer only with the option number. Do not include any explanation or reasoning.
                Question: {question}
                Options: {options}
                Answer:"""
    return prompt

def get_input(prompt, MODEL_NAME, model, tokenizer):
    # formated_example = format_prompt(example)

    if MODEL_NAME=='Qwen/Qwen3-4B-Base':
        prompt = prompt
    else:
        chat = [
            {'role': 'system', 'content': 'You are a helpful and precise assistant. Answer the MCQ questions accurately with one of the options provided. Answer only with the option number. Do not include any explanation or reasoning.'},
            {"role": "user", "content": f"{prompt}"}
        ]
        prompt = tokenizer.apply_chat_template(chat,  tokenize=False,     add_generation_prompt=True,    enable_thinking=False )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    return inputs


def get_response_with_logits(model_inputs, options, tokenizer, model):

    outputs = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        do_sample=False,
        # temperature=0.7,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_ids = outputs.sequences
    input_len = model_inputs["input_ids"].shape[-1]

    output_ids = generated_ids[0][input_len:].tolist()

        # parsing thinking content
    # try:
    #     # rindex finding 151668 (</think>)
    #     index = len(output_ids) - output_ids[::-1].index(151668)
    # except ValueError:
    #     index = 0

    # thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    # content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    response = tokenizer.decode(output_ids, skip_special_tokens=True)
    # print(f"Raw response: {response}")
    # print(f"Content after thinking: {content}")
    # print(f"Token IDs of response: {output_ids}")
    option = re.findall(r'\d+', response)[-1]
    # print(f"options: {option}")

    # response = response.strip().split()[0]  # Get the first token of the response
    option_token = tokenizer.encode(option, add_special_tokens=False)
    output_id_of_option = output_ids.index(option_token[0])
    logits = outputs.scores[output_id_of_option]   # (1, vocab_size)
    log_probs = F.log_softmax(logits, dim=-1)

    scores = {}

    for opt in ['1','2','3','4']:
        token_ids = tokenizer.encode(opt, add_special_tokens=False)
        scores[opt] = log_probs[0, token_ids[0]].item()
    # print(scores)
    return option, scores

In [ ]:
dataset = datasets.load_dataset("cais/mmlu", 'all')

In [ ]:
subject_count = {}
college_level = {}
high_school_level = {}
for example in dataset['test']:
    subject = example['subject']
    subject_count[subject] = subject_count.get(subject, 0) + 1
    if subject.startswith('college'):
        college_level[subject] = college_level.get(subject, []) + [example]
    elif subject.startswith('high_school'):
        high_school_level[subject] = high_school_level.get(subject, []) + [example]

In [ ]:
# MODELS = ['Qwen/Qwen3-4B-Base','Qwen/Qwen3-4B-Instruct-2507',  ]

MODELS = ['Qwen/Qwen3-4B-Instruct-2507', 'Qwen/Qwen3-4B-Base']

save_mcq_results = {}

for MODEL_NAME in MODELS:
    random.seed(42)
    # Load tokenizer + model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="mps",
    )

    subsampled_mmlu_high_school = []
    subject_scores = {subject: 0 for subject in high_school_level.keys()}
    for subject, examples in high_school_level.items():
        subset = random.sample(examples, 8)
        subsampled_mmlu_high_school.extend(subset)

    correct = 0
    total = 0

    for example in tqdm(subsampled_mmlu_high_school):
        question,choices = example["question"], example["choices"]
        prompt = format_prompt(question, choices)
        model_inputs = get_input(prompt, MODEL_NAME, model, tokenizer)

        response, prob = get_response_with_logits(model_inputs, choices, tokenizer, model)
        gold = example["answer"]
        scores = []
        print(int(response.strip()), gold)
        if int(response.strip()) == gold:
            
            correct += 1
            subject_scores[example['subject']] += 1

        total += 1
        scores.append(prob)
        

    accuracy = correct / total
    subject_accuracies = {subject: score/20 for subject, score in subject_scores.items()}
    print(f"College-level Accuracy: {accuracy:.4f}")
    print(f"Subject Accuracies: {subject_accuracies}")
    save_mcq_results[MODEL_NAME] = {"accuracy": accuracy, "subject_accuracies": subject_accuracies, 'scores': scores}


    # subsampled_mmlu_high_school = []
    # for subject, examples in high_school_level.items():
    #     subset = random.sample(examples, 20)
    #     subsampled_mmlu_high_school.extend(subset)
    # correct = 0
    # total = 0

    # for example in tqdm(subsampled_mmlu_high_school):
    #     question,choices = example["question"], example["choices"]
    #     prompt = format_prompt(question, choices)
    #     model_inputs = get_input(prompt, MODEL_NAME, model, tokenizer)

    #     response, prob = get_response_with_logits(model_inputs, choices, tokenizer, model)
    #     gold = example["answer"]
    #     scores = []
    #     if int(response.strip()) == gold:
    #         correct += 1

    #     total += 1
    #     scores.append(prob)

    # accuracy = correct / total
    # print(f"High-school Accuracy: {accuracy:.4f}")  
    # save_mcq_results[MODEL_NAME]["high_school"] = accuracy